# SpectraShift Week 5: seed 17
Run the 15 M0-M4 anchor jobs sequentially. Attach the source, Week 2 data, Week 4 aggregate, Week 5 contracts/pilots, and `spectrashift-week4-seed17`. Use GPU T4 x2 with Internet off.


In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week5.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 5 source bundle, found {bundles}'
    source_work = Path('/kaggle/working/week5-source')
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 5 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
SEED = 17

WORK = Path('/kaggle/working/spectrashift-week5-seed17')
WORK.mkdir(parents=True, exist_ok=True)
manifests = sorted(INPUT.rglob('partitions.parquet'))
normalizations = sorted(INPUT.rglob('normalization.json'))
freeze_summaries = sorted(INPUT.rglob('freeze_summary.json'))
contracts_summaries = sorted(INPUT.rglob('week5_contracts_summary.json'))
week4_summaries = sorted(INPUT.rglob('week4_run_summary.json'))
assert len(manifests) == len(normalizations) == len(freeze_summaries) == len(contracts_summaries) == len(week4_summaries) == 1
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
CONTRACTS_SUMMARY = contracts_summaries[0]
CONTRACTS = json.loads(CONTRACTS_SUMMARY.read_text())
from spectrashift.train.common import file_sha256
assert file_sha256(week4_summaries[0]) == CONTRACTS['week4_summary_sha256']
config = yaml.safe_load((PROJECT / 'configs/downstream/week5.yaml').read_text())
config['data']['manifest_path'] = str(manifests[0])
config['data']['staged_root'] = str(STAGED)
config['data']['normalization_path'] = str(normalizations[0])
config['data']['freeze_summary_path'] = str(freeze_summaries[0])
RUNTIME_CONFIG = WORK / 'week5.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
import torch
encoder_paths = {}
for path in INPUT.rglob('encoder-final.pt'):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if payload.get('model_id') in {'M2','M3','M4'}:
        encoder_paths[str(payload['run_id'])] = path
print({'project': str(PROJECT), 'work': str(WORK), 'encoders': sorted(encoder_paths)})


In [ ]:
assert torch.cuda.is_available(), 'Select GPU T4 x2 before running'
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
assert 'T4' in gpu_name and f'sm_{major}{minor}' in torch.cuda.get_arch_list()
expected = {f'week4-{model}-seed17' for model in ('m2','m3','m4')}
assert set(encoder_paths) == expected, f'Expected {expected}, found {set(encoder_paths)}'
pilot_summaries = sorted(INPUT.rglob('week5_pilot_summary.json'))
assert len(pilot_summaries) == 1
PILOT_SUMMARY = pilot_summaries[0]
print({'gpu': gpu_name, 'gpu_count': torch.cuda.device_count()})


In [ ]:
from spectrashift.train.week5 import run_week5_seed

summary = run_week5_seed(
    RUNTIME_CONFIG, SEED, WORK, CONTRACTS_SUMMARY, PILOT_SUMMARY,
    encoder_paths, resume_roots=[INPUT],
)
print(json.dumps(summary, indent=2))
assert summary['week5_seed_complete'] and summary['run_count'] == 15
assert len(summary['feature_caches']) == 4 and summary['evaluation_labels_loaded'] is False
